In [8]:
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("Load Database")
    .getOrCreate()
)
spark 

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 12:33:23 WARN Utils: Your hostname, ammar-malik-Victus-by-HP-Gaming-Laptop-15-fa0xxx, resolves to a loopback address: 127.0.1.1; using 192.168.1.25 instead (on interface wlp0s20f3)
26/06/16 12:33:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 12:33:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 12:33:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/16 12:33:25 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [10]:
daily_summary = spark.read.parquet("../Data/03_Gold/daily_weather_summary")

In [11]:
daily_pd = daily_summary.toPandas() 

In [12]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="student",
    password="1234",
    database="weather_warehouse"
)

cursor = conn.cursor()

In [13]:
for _, row in daily_pd.iterrows():
    cursor.execute("""
        INSERT INTO daily_weather_summary
        VALUES (%s,%s,%s,%s,%s)
    """, (
        row["weather_date"],
        float(row["avg_temperature"]),
        float(row["max_temperature"]),
        float(row["min_temperature"]),
        float(row["avg_humidity"])
    ))

conn.commit()

In [15]:
hourly_trend = spark.read.parquet("../Data/03_Gold/hourly_temperature_trend")
hourly_pd = hourly_trend.toPandas()

extreme_weather = spark.read.parquet("../Data/03_Gold/extreme_weather_report")
extreme_pd = extreme_weather.toPandas() 

In [16]:
for _, row in hourly_pd.iterrows():
    cursor.execute("""
        INSERT INTO hourly_temperature_trend
        VALUES (%s,%s)
    """, (
        int(row["weather_hour"]),
        float(row["avg_temperature"])
    ))

conn.commit()

In [17]:
for _, row in extreme_pd.iterrows():
    cursor.execute("""
        INSERT INTO extreme_weather_report
        VALUES (%s,%s,%s,%s,%s)
    """, (
        row["weather_date"],
        float(row["avg_temperature"]),
        float(row["max_temperature"]),
        float(row["min_temperature"]),
        float(row["avg_humidity"])
    ))

conn.commit()